<a href="https://colab.research.google.com/github/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/blob/main/Preprocessing_Notebook_4th_edition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Pre-processing Methodology

## Executive Summary
In the film industry (as in software product management), capital allocation is the ultimate challenge. Studios must decide whether to invest hundreds of millions of dollars into a project long before the final product exists.

This project shifts the focus from descriptive analytics to a **Decision Support System**. Our "North Star" is strictly financial: we are building a predictive machine learning model (Random Forest) to assess the probability of a project achieving a positive Return on Investment (ROI) based entirely on pre-production constraints and historical attributes.

# 1. Project Introduction & Preprocessing Methodology: The Greenlight Algorithm 🚦

## The 4 Pillars of Preprocessing
To prepare our raw data for the Greenlight Algorithm, we will adhere to a rigorous, production-grade preprocessing pipeline divided into four phases:

### I. Data Cleaning & Integrity (The Foundation)
Before any modeling occurs, we must ensure the dataset is structurally sound and logically possible.
* **Sanity Checks & Missing Values:** Financial models cannot tolerate placeholder data. For example, reported budgets of `$0` are physically impossible and will be strictly converted to `NaN` to prevent mathematical errors in ROI calculations. We'll check data types (and adjust them if needed), handle NA's, and filter out physically impossible data.
* **Single Source of Truth (SSOT):** When merging multiple datasets (e.g., TMDB, The Numbers), we will resolve overlapping columns (like `budget` vs. `production_budget`) by defining and documenting a clear SSOT based on data completeness and precision.

### II. Financial Standardization (Macroeconomics)
Time-series financial data contains inherent macroeconomic bias. A `$100M` budget in 1990 carries vastly different risk than a `$100M` budget in 2024.
* **Real vs. Nominal Dollars:** All financial metrics (Budget, Gross Revenue) will be adjusted to **Real 2024 Dollars** using historical Consumer Price Index (CPI) data. This ensures the algorithm evaluates risk on a level playing field across decades.

### III. Feature Engineering (Signals & NLP)
Raw data (names, text, dates) must be translated into mathematical "signals" based on the **R.I.C.E. Product Framework** (Reach, Impact, Confidence, Effort) that will lead us through all the analysis steps:

* **Confidence Signals (Historical Authority):** Calculating pre-release "Win Rates" for Directors, Writers, and Stars to proxy execution risk.

* **Reach Signals (Natural Language Processing):** Utilizing NLP (`CountVectorizer`) on plot overviews to mathematically extract and flag high-value semantic themes (e.g., "Family", "War") rather than relying on manual keyword searches.

### IV. Target Formulation & Leakage Prevention (The North Star)
The golden rule of predictive modeling is preventing the model from "cheating" by seeing the future.

* **Financial Target:** Our target variable is strictly financial classifier (`is_profitable`), derived from real revenue and real budget.

* **Leakage Prevention:** Aspirational metrics like `imdb_score` or `vote_average` will *not* be used as targets. Instead, historical proxy scores will be used as pre-release risk multipliers, ensuring a strict boundary between pre-launch features and post-launch outcomes.


---

# Loading Data

In [ ]:
import pandas as pd

# Define a compact Column Summary Function for checking data integrity (types, NA%, and NA count )
def quick_column_summary(df, table_name):
    print(f"\n📋 Column Summary for `{table_name}`\n")
    total_rows = len(df)
    summary = pd.DataFrame({
        'Column': df.columns,
        'Data Type': [df[col].dtype for col in df.columns],
        'NA Count': [df[col].isna().sum() for col in df.columns],
        '% Missing': [round(df[col].isna().mean() * 100, 2) for col in df.columns]
    })
    display(summary)

## 1st Dataset: Movie Franchises



In [ ]:
# 1. Movie Data Analysis Dataset (Base Table)
!wget https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/movie.csv -O movie.csv

movie_franchises = pd.read_csv("movie.csv")

# Rename the IMDB score column (Kept for potential EDA, but no longer our target)
movie_franchises = movie_franchises.rename(columns={"score": "imdb_score"})

--2026-03-13 19:32:52--  https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/movie.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1294548 (1.2M) [text/plain]
Saving to: ‘movie.csv’

movie.csv           100%[===================>]   1.23M  --.-KB/s    in 0.008s  

2026-03-13 19:32:52 (151 MB/s) - ‘movie.csv’ saved [1294548/1294548]



### First check

In [ ]:
# Display the first few rows
movie_franchises.head()

,name,rating,genre,year,released,imdb_score,votes,director,writer,star,country,budget,gross,company,runtime
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000.0,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000.0,46998772.0,Warner Bros.,146.0
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000.0,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000.0,58853106.0,Columbia Pictures,104.0
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000.0,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000.0,538375067.0,Lucasfilm,124.0
3,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.7,221000.0,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000.0,83453539.0,Paramount Pictures,88.0
4,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.3,108000.0,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000.0,39846344.0,Orion Pictures,98.0


In [ ]:
# Display the dataset shape
movie_franchises.shape

(5436, 15)

In [ ]:
# --- FINANCIAL NORTH STAR ADJUSTMENTS ---

import pandas as pd
import numpy as np

# Fix 1: Treat $0 as Missing Data (Data Integrity)
movie_franchises['budget'] = movie_franchises['budget'].replace(0, np.nan)
movie_franchises['gross'] = movie_franchises['gross'].replace(0, np.nan)

# Fix 2: Omit observations with NA's in FINANCIAL target variables ONLY
movie_franchises = movie_franchises[
    movie_franchises['budget'].notna() &
    movie_franchises['gross'].notna()
].copy()

In [ ]:
# Check for data types and NA's
quick_column_summary(movie_franchises, 'movie_franchises')


📋 Column Summary for `movie_franchises`



,Column,Data Type,NA Count,% Missing
0,name,object,0,0.00
1,rating,object,12,0.22
2,genre,object,0,0.00
3,year,int64,0,0.00
4,released,object,0,0.00
5,imdb_score,float64,0,0.00
6,votes,float64,0,0.00
7,director,object,0,0.00
8,writer,object,0,0.00
9,star,object,0,0.00


In [ ]:
# Rename the IMDB score column
movie_franchises = movie_franchises.rename(columns={"score": "imdb_score"})  # replace "score" with our desirable target name - "imdb_score"

In [ ]:
# Omit observations with NA's in target variables
movie_franchises = movie_franchises[
    movie_franchises['budget'].notna() &
    movie_franchises['gross'].notna()
].copy()

### 📥 Table Acquisition Summary: `movie_franchises`

#### 🎯 Relevant Variables

| Column         | Description                     | Relevance                          |
|----------------|----------------------------------|-------------------------------------|
| `name`         | Movie name (key)                | ✅ Unique ID across datasets         |
| `imdb_score`   | IMDB rating                     | ✅ Target variable #1                |
| `budget`       | Budget in dollars               | 📌 Required for ROI (target #2)     |
| `gross`        | Revenue in dollars              | 📌 Required for ROI (target #2)     |
| `votes`        | Number of user ratings          | 🧪 May influence IMDB score         |
| `genre`, `rating`, `year`, `released` | Movie metadata | 📊 Potential features |
| `director`, `writer`, `star`, `company` | People / studio involved | 📊 Potential features |
| `runtime`      | Duration in minutes             | 📊 Feature (e.g., audience fatigue) |
| `country`      | Country of production           | 📊 Feature for cultural reception   |

## 2nd Dataset: additional Movie Franchises

In [ ]:
# 2. Global Movie Franchise Revenue and Budget Data

!wget https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/MovieFranchises.csv -O MovieFranchises.csv
import pandas as pd
data2 = pd.read_csv("MovieFranchises.csv") # Save in a different name due to similar name to the 1st dataset

--2026-03-13 19:32:59--  https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/MovieFranchises.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26322 (26K) [text/plain]
Saving to: ‘MovieFranchises.csv’

MovieFranchises.csv 100%[===================>]  25.71K  --.-KB/s    in 0.001s  

2026-03-13 19:32:59 (26.9 MB/s) - ‘MovieFranchises.csv’ saved [26322/26322]



### First check

In [ ]:
# Display the first few rows
data2.head()

,index,MovieID,Title,Lifetime Gross,Year,Studio,Rating,Runtime,Budget,ReleaseDate,VoteAvg,VoteCount,FranchiseID
0,0,1001,Star Wars: Episode IV - A New Hope,775398007,1977,Lucasfilm,PG,121.0,11000000.0,05-25-77,4.09,96233.0,101.0
1,1,1002,Star Wars: Episode V - The Empire Strikes Back,538375067,1980,Lucasfilm,PG,124.0,18000000.0,06-20-80,4.12,79231.0,101.0
2,2,1003,Star Wars: Episode VI - Return of the Jedi,475106177,1983,Lucasfilm,PG,135.0,32500000.0,05-25-83,3.98,76082.0,101.0
3,3,1004,Jurassic Park,1109802321,1993,Universal Pictures,PG-13,127.0,63000000.0,06-11-93,3.69,82700.0,102.0
4,4,1005,The Lost World: Jurassic Park,618638999,1997,Universal Pictures,PG-13,129.0,73000000.0,05-23-97,3.01,19721.0,102.0


In [ ]:
# Display the dataset shape
data2.shape

(605, 13)

In [ ]:
# Check for data types and NA's
quick_column_summary(data2, 'data2')


📋 Column Summary for `data2`



,Column,Data Type,NA Count,% Missing
0,index,int64,0,0.00
1,MovieID,object,0,0.00
2,Title,object,0,0.00
3,Lifetime Gross,object,0,0.00
4,Year,object,539,89.09
5,Studio,object,545,90.08
6,Rating,object,545,90.08
7,Runtime,float64,545,90.08
8,Budget,float64,545,90.08
9,ReleaseDate,object,545,90.08


In [ ]:
# Keep only the useful parts of data2
data2 = data2[['MovieID', 'Title', 'Lifetime Gross']].copy()

### 📥 Table Acquisition Summary: `data2`

This table had lots of missing values. But still, the table includes financial data that can add us more information about our target variable ROI.

#### 🔁 Remaining Variables

| `movie_franchises` | `studio_financials` | Action |
|--------------------|---------------------|--------|
| `name`             | `Title`             | Normalize to `movie_id` for matching |
| `budget`           | `Budget`            | Compare and retain best version |
| `gross`            | `Lifetime Gross`    | Compare with `gross` |


## 3rd Dataset: TMDB data

In [ ]:
# If the 3rd dataset have error contains "LocalFileSystem is not supported" then use the code:
# pip install -U datasets

In [ ]:
# 3. TMDB 5000 Movies Dataset

!pip install datasets

from datasets import load_dataset
import pandas as pd

# Load the TMDB dataset from Hugging Face
dataset = load_dataset("AiresPucrs/tmdb-5000-movies", split="train")
tmdb_data = pd.DataFrame(dataset)

# Save the DataFrame to a CSV file
tmdb_data.to_csv("tmdb_movies.csv", index=False)

# Confirm the file exists in the current directory
import os
os.listdir()

['.config',
 'movies.csv',
 'drive',
 'MovieFranchises.csv',
 'final_dataset.csv',
 'tmdb_movies.csv',
 'movie.csv',
 'sample_data']

### First Check

In [ ]:
# Display the first few rows
tmdb_data.head()

,id,budget,genres,homepage,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,5,4000000,"[{""id"": 80, ""name"": ""Crime""}, {""id"": 35, ""name...",None,"[{""id"": 612, ""name"": ""hotel""}, {""id"": 613, ""na...",en,Four Rooms,It's Ted the Bellhop's first night on the job....,22.876230,"[{""name"": ""Miramax Films"", ""id"": 14}, {""name"":...",...,4300000,98.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Twelve outrageous guests. Four scandalous requ...,Four Rooms,6.5,530,"[{""cast_id"": 42, ""character"": ""Ted the Bellhop...","[{""credit_id"": ""52fe420dc3a36847f800012d"", ""de..."
1,11,11000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",http://www.starwars.com/films/star-wars-episod...,"[{""id"": 803, ""name"": ""android""}, {""id"": 4270, ...",en,Star Wars,Princess Leia is captured and held hostage by ...,126.393695,"[{""name"": ""Lucasfilm"", ""id"": 1}, {""name"": ""Twe...",...,775398007,121.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"A long time ago in a galaxy far, far away...",Star Wars,8.1,6624,"[{""cast_id"": 3, ""character"": ""Luke Skywalker"",...","[{""credit_id"": ""52fe420dc3a36847f8000437"", ""de..."
2,12,94000000,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 10751...",http://movies.disney.com/finding-nemo,"[{""id"": 494, ""name"": ""father son relationship""...",en,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",85.688789,"[{""name"": ""Pixar Animation Studios"", ""id"": 3}]",...,940335536,100.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"There are 3.7 trillion fish in the ocean, they...",Finding Nemo,7.6,6122,"[{""cast_id"": 8, ""character"": ""Marlin (voice)"",...","[{""credit_id"": ""52fe420ec3a36847f80006b1"", ""de..."
3,13,55000000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",None,"[{""id"": 422, ""name"": ""vietnam veteran""}, {""id""...",en,Forrest Gump,A man with a low IQ has accomplished great thi...,138.133331,"[{""name"": ""Paramount Pictures"", ""id"": 4}]",...,677945399,142.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"The world will never be the same, once you've ...",Forrest Gump,8.2,7927,"[{""cast_id"": 7, ""character"": ""Forrest Gump"", ""...","[{""credit_id"": ""52fe420ec3a36847f800076b"", ""de..."
4,14,15000000,"[{""id"": 18, ""name"": ""Drama""}]",http://www.dreamworks.com/ab/,"[{""id"": 255, ""name"": ""male nudity""}, {""id"": 29...",en,American Beauty,"Lester Burnham, a depressed suburban father in...",80.878605,"[{""name"": ""DreamWorks SKG"", ""id"": 27}, {""name""...",...,356296601,122.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Look closer.,American Beauty,7.9,3313,"[{""cast_id"": 6, ""character"": ""Lester Burnham"",...","[{""credit_id"": ""52fe420ec3a36847f8000809"", ""de..."


In [ ]:
# Display the dataset shape
tmdb_data.shape

(4803, 22)

In [ ]:
# Check for data types and NA's
quick_column_summary(tmdb_data, 'tmdb_data')


📋 Column Summary for `tmdb_data`



,Column,Data Type,NA Count,% Missing
0,id,int64,0,0.00
1,budget,int64,0,0.00
2,genres,object,0,0.00
3,homepage,object,3091,64.36
4,keywords,object,0,0.00
5,original_language,object,0,0.00
6,original_title,object,0,0.00
7,overview,object,3,0.06
8,popularity,float64,0,0.00
9,production_companies,object,0,0.00


### 📥 Table Acquisition: `tmdb_data`

This is the richest and most structured table so far. It includes both structured and nested (JSON-like) data, contributing heavily to both prediction targets.

---

#### 🎯 Relevant Variables

| Column | Description | Relevance |
|--------|-------------|-----------|
| `title` | Movie name | ✅ Used to create `movie_id` |
| `vote_average` | Average audience rating | ✅ Proxy for IMDB score |
| `vote_count` | Number of votes | 🧪 May influence or complement score |
| `budget` | Production cost | 📌 Required for ROI |
| `revenue` | Box office revenue | 📌 Required for ROI |
| `runtime` | Duration in minutes | 📊 Feature for pacing / cost |
| `popularity` | TMDB popularity score | 📊 Social visibility |
| `release_date` | Date released | 📊 Use for time features (month, year) |
| `genres` | List of genres (JSON) | 🧠 To parse later for genre-based analysis |
| `keywords` | Thematic keywords (JSON) | 🧠 Useful after parsing |
| `overview`, `tagline` | Textual summary & tagline | 🧠 Potential for NLP sentiment modeling |
| `original_language` | Language code (e.g., 'en') | 📊 Cultural/demographic indicator |
| `production_companies` | Companies involved (JSON) | 🧠 Feature engineering (studio power) |
| `production_countries` | Countries involved (JSON) | 📊 International impact |
| `spoken_languages` | Languages spoken (JSON) | 📊 Audience reach |
| `cast`, `crew` | Cast and crew (JSON) | 🧠 Feature-rich, parse later |
| `status` | e.g., Released, Post-production, etc. | 🧪 May correlate with box office |

---

#### 🧠 Summary

- This table contributes to both `imdb_score_features` and `roi_features`
- Contains multiple nested fields that will be parsed during feature engineering
- Will be save in SQLite as `raw_tmdb_data`


## 4th Dataset: Meta-Analysis Data

In [ ]:
# 4. Complete Movie Metadata Dataset

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
file_path = '/content/drive/My Drive/Projects/Blockbuster Movies/movies.csv'  # Adjust path as needed
meta_data = pd.read_csv(file_path)

# Save the DataFrame to a CSV file
meta_data.to_csv("movies.csv", index=False)

# Confirm the file exists in the current directory
import os
os.listdir()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['.config',
 'movies.csv',
 'drive',
 'MovieFranchises.csv',
 'final_dataset.csv',
 'tmdb_movies.csv',
 'movie.csv',
 'sample_data']

### First Check

In [ ]:
# Display the first few rows
meta_data.head()

,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations
0,615656,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000.0,3.520565e+08,116.0,Released,Back for seconds.,7.079,1365.0,Jason Statham-Wu Jing-Shuya Sophia Cai-Sergio ...,based on novel or book-sequel-kaiju,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,/qlxy8yo5bcgUw2KAmmojUKp4rHd.jpg,1006462-298618-569094-1061181-346698-1076487-6...
1,758323,The Pope's Exorcist,Horror-Mystery-Thriller,en,Father Gabriele Amorth Chief Exorcist of the V...,5953.227,Screen Gems-2.0 Entertainment-Jesus & Mary-Wor...,2023-04-05,18000000.0,6.567582e+07,103.0,Released,Inspired by the actual files of Father Gabriel...,7.433,545.0,Russell Crowe-Daniel Zovatto-Alex Essoe-Franco...,spain-rome italy-vatican-pope-pig-possession-c...,/9JBEPLTPSm0d1mbEcLxULjJq9Eh.jpg,/hiHGRbyTcbZoLsYYkO4QiCLYe34.jpg,713704-296271-502356-1076605-1084225-1008005-9...
2,533535,Deadpool & Wolverine,Action-Comedy-Science Fiction,en,A listless Wade Wilson toils away in civilian ...,5410.496,Marvel Studios-Maximum Effort-21 Laps Entertai...,2024-07-24,200000000.0,1.326387e+09,128.0,Released,Come together.,7.765,3749.0,Ryan Reynolds-Hugh Jackman-Emma Corrin-Matthew...,hero-superhero-anti hero-mutant-breaking the f...,/8cdWjvZQUExUUTzyp4t6EDMubfO.jpg,/dvBCdCohwWbsP5qAaglOXagDMtk.jpg,573435-519182-957452-1022789-945961-718821-103...
3,667538,Transformers: Rise of the Beasts,Action-Adventure-Science Fiction,en,When a new threat capable of destroying the en...,5409.104,Skydance-Paramount-di Bonaventura Pictures-Bay...,2023-06-06,200000000.0,4.070455e+08,127.0,Released,Unite or fall.,7.340,1007.0,Anthony Ramos-Dominique Fishback-Luna Lauren V...,peru-alien-end of the world-based on cartoon-b...,/gPbM0MK8CP8A174rmUwGsADNYKD.jpg,/woJbg7ZqidhpvqFGGMRhWQNoxwa.jpg,496450-569094-298618-385687-877100-598331-4628...
4,693134,Dune: Part Two,Science Fiction-Adventure,en,Follow the mythic journey of Paul Atreides as ...,4742.163,Legendary Pictures,2024-02-27,190000000.0,6.838137e+08,167.0,Released,Long live the fighters.,8.300,2770.0,Timothée Chalamet-Zendaya-Rebecca Ferguson-Jav...,epic-based on novel or book-fight-sandstorm-sa...,/czembW0Rk1Ke7lCJGahbOhdCuhV.jpg,/xOMo8BRK7PfcJv9JCnx7s5hj0PX.jpg,438631-763215-792307-1011985-467244-634492-359...


In [ ]:
# Display the dataset shape
meta_data.shape

(722317, 20)

In [ ]:
# Count rows where budget is 0
zero_budget_count = (meta_data['budget'] == 0).sum()
print(f"Number of rows with budget = 0: {zero_budget_count}")

# Count rows where revenue is 0
zero_revenue_count = (meta_data['revenue'] == 0).sum()
print(f"Number of rows with revenue = 0: {zero_revenue_count}")

Number of rows with budget = 0: 685547
Number of rows with revenue = 0: 705113


In [ ]:
# Check for data types and NA's
quick_column_summary(meta_data, 'meta_data')


📋 Column Summary for `meta_data`



,Column,Data Type,NA Count,% Missing
0,id,int64,0,0.00
1,title,object,6,0.00
2,genres,object,210317,29.12
3,original_language,object,0,0.00
4,overview,object,118243,16.37
5,popularity,float64,0,0.00
6,production_companies,object,384926,53.29
7,release_date,object,51549,7.14
8,budget,float64,0,0.00
9,revenue,float64,0,0.00


### 📥 Table Acquisition: `meta_data`

This dataset appears to be an updated or complementary version of `tmdb_data`, containing recent and upcoming titles with similar structure.

---

#### 🎯 Relevant Variables

| Column | Description | Relevance |
|--------|-------------|-----------|
| `title` | Movie title | ✅ Used to create `movie_id` |
| `vote_average` / `vote_count` | User rating and count | ✅ Score-related |
| `budget`, `revenue` | Financial data | 📌 Used for ROI |
| `runtime`, `release_date` | Timing & length | 📊 Influences score & ROI |
| `popularity` | TMDB popularity score | 📊 Social reach |
| `genres`, `keywords`, `overview`, `tagline` | Text / tags | 🧠 Feature-rich, parse later |
| `original_language` | Language code | 📊 Cultural signal |
| `status` | Release status | 🧪 Could correlate with results |
| `production_companies` | Studios involved | 🧠 To group studio trends |
| `credits` | Raw cast and crew | 🧠 To parse later for influence modeling |

---

#### 🧠 Summary

- High overlap with `tmdb_data` (Table 3) — strong candidate for integration
- Contributes to both `imdb_score_features` and `roi_features`
- Will require deduplication and possible enrichment during post-acquisition phase
- Will be save in SQLite as `meta_data`


## 5th Dataset: Revenues Data

In [ ]:
# 5. Movie Revenue Analysis Dataset

!wget https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/final_dataset.csv -O final_dataset.csv
import pandas as pd
financial_data = pd.read_csv("final_dataset.csv")

--2026-03-13 19:34:03--  https://raw.githubusercontent.com/JohnnySolo/Data-Analysis-Project---Blockbuster-Movies/main/final_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 456039 (445K) [text/plain]
Saving to: ‘final_dataset.csv’

final_dataset.csv   100%[===================>] 445.35K  --.-KB/s    in 0.007s  

2026-03-13 19:34:03 (61.9 MB/s) - ‘final_dataset.csv’ saved [456039/456039]



### First Check

In [ ]:
# Display the first few rows
financial_data.head()

,Unnamed: 0,movie,year,production_budget,domestic_gross,foreign_gross,worldwide_gross,month,profit,profit_margin,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
0,0,Avatar,2009,425000000,760507625,2015837654,2776345279,12,2351345279,0.846921,...,0,0,0,0,0,1,0,0,0,0
1,1,Pirates of the Caribbean: On Stranger Tides,2011,410600000,241063875,804600000,1045663875,5,635063875,0.607331,...,0,0,0,0,0,0,0,0,0,0
2,2,Avengers: Age of Ultron,2015,330600000,459005868,944008095,1403013963,5,1072413963,0.764364,...,0,0,0,0,0,1,0,0,0,0
3,3,Avengers: Infinity War,2018,300000000,678815482,1369318718,2048134200,4,1748134200,0.853525,...,0,0,0,0,0,0,0,0,0,0
4,4,Justice League,2017,300000000,229024295,426920914,655945209,11,355945209,0.542645,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
# Display the dataset shape
financial_data.shape

(1759, 39)

In [ ]:
# Check for data types and NA's
quick_column_summary(financial_data, 'financial_data')


📋 Column Summary for `financial_data`



,Column,Data Type,NA Count,% Missing
0,Unnamed: 0,int64,0,0.00
1,movie,object,0,0.00
2,year,int64,0,0.00
3,production_budget,int64,0,0.00
4,domestic_gross,int64,0,0.00
5,foreign_gross,int64,0,0.00
6,worldwide_gross,int64,0,0.00
7,month,int64,0,0.00
8,profit,int64,0,0.00
9,profit_margin,float64,0,0.00


### 📥 Table Acquisition: `financial_data`

This table is highly focused on financial metrics and genre distribution. It provides engineered columns for ROI, profit, and genre flags, making it very valuable for prediction.

---

#### 🎯 Relevant Variables

| Column | Description | Relevance |
|--------|-------------|-----------|
| `movie` | Movie name | ✅ Used to create `movie_id` |
| `production_budget`, `domestic_gross`, `foreign_gross`, `worldwide_gross` | Raw inputs for ROI | ✅ |
| `profit`, `roi`, `profit_margin`, `pct_foreign` | Pre-calculated finance metrics | ✅ |
| `vote_average`, `vote_count`, `popularity` | Score-related audience signals | ✅ |
| `original_language`, `release_date`, `month` | Contextual/cultural features | ✅ |
| `Action`, `Drama`, etc. | Binary genre flags | ✅ Helps both score and ROI models |

---

#### 🧠 Summary

- Strongest financial data table (calculated ROI & profit)
- Includes one-hot encoded genre info (clean and ready)
- Will contribute to both `imdb_score_features` and `roi_features`
- Saved in SQLite as `raw_financial_data`


---

# I. Data Cleaning & Integrity

## Introduction: Data Consolidation & The SSOT Strategy

Merging five distinct datasets presents a significant data engineering challenge: overlapping columns (e.g., multiple `budget` columns) and inconsistent coverage.

To resolve this, we will execute a **Single Source of Truth (SSOT)** merging strategy designed to maximize our pool of financially valid movies without introducing duplicates or corrupted data.

### 1. Normalizing Identifiers (The Join Key)
To prepare for the merge, we create a primary key called `movie_id` across all datasets.
* **Action:** We extract the movie title, convert it to lowercase, and strip leading/trailing whitespace.
* **Safety Check:** We drop duplicate `movie_id`s within each individual dataset *before* merging to prevent an outer join "explosion" (e.g., accidentally creating 50 rows for movies with common names like "Cinderella").

### 2. The Merging Strategy: Outer Join
Instead of a restrictive Left Join, we utilize an **Outer Join** on the datasets containing primary financial data (`movie_franchises`, `financial_data`, `tmdb_data`, `meta_data`).
* **The "Why":** A Left Join would artificially cap our dataset size to the ~5,400 rows of the base table. By outer joining, we capture every unique movie across all sources, allowing us to evaluate a much larger potential universe of films.

### 3. Establishing the Single Source of Truth (SSOT)
After the outer join, we are left with multiple conflicting columns for the same metric (e.g., `budget`, `production_budget`, `budget_tmdb`).
* **The Coalesce Technique:** We use Pandas' `combine_first()` to establish an SSOT. We prioritize the most reliable data source (e.g., `production_budget`), and if that value is missing (`NaN`), the algorithm automatically cascades down to the next available source (`budget_base`, then `budget_tmdb`, etc.) until a valid number is found.
* **Result:** We create two master columns: `ultimate_budget` and `ultimate_revenue`.

### 4. The Financial North Star (The Great Purge)
Because our algorithm's strict goal is predicting financial ROI, any row missing an `ultimate_budget` or `ultimate_revenue` is structurally useless.
* **Action:** We drop any movie that lacks complete financial data across all sources. This aggressively filters the massive outer-joined dataset down to only the absolute highest-quality, financially verified records.



### 1. Normalizing Identifiers (The Join Key)

In [ ]:
# 1. Normalize Titles
def normalize_title(df, title_col):
    df['movie_id'] = df[title_col].astype(str).str.strip().str.lower()
    return df.drop_duplicates(subset=['movie_id'])

In [ ]:
# Apply normalization and deduplication to all datasets
movie_franchises = normalize_title(movie_franchises, 'name')
data2 = normalize_title(data2, 'Title')
tmdb_data = normalize_title(tmdb_data, 'title')
meta_data = normalize_title(meta_data, 'title')
financial_data = normalize_title(financial_data, 'movie')

In [ ]:
# 1. Identify the pool of movies we ALREADY have from our main financial tables
base_movie_ids = set(movie_franchises['movie_id']).union(set(financial_data['movie_id']))
print(f"Unique movies in our base financial tables: {len(base_movie_ids)}")

# 2. Filter the external datasets to ONLY rows with actual money (No zeroes, no NaNs)
valid_meta = meta_data[(meta_data['budget'] > 0) & (meta_data['revenue'] > 0)]
valid_tmdb = tmdb_data[(tmdb_data['budget'] > 0) & (tmdb_data['revenue'] > 0)]

# 3. Calculate the NET NEW movies they bring to the table
new_from_meta = set(valid_meta['movie_id']) - base_movie_ids
new_from_tmdb = set(valid_tmdb['movie_id']) - base_movie_ids - new_from_meta # Exclude ones meta already added

print(f"\nOut of 722,000 rows, meta_data has {len(valid_meta)} financially valid movies.")
print(f"👉 NET NEW movies added by meta_data: {len(new_from_meta)}")

print(f"\nOut of 4,803 rows, tmdb_data has {len(valid_tmdb)} financially valid movies.")
print(f"👉 NET NEW movies added by tmdb_data: {len(new_from_tmdb)}")

# The absolute maximum valid universe size:
total_universe = len(base_movie_ids) + len(new_from_meta) + len(new_from_tmdb)
print(f"\n🚀 Absolute Maximum Viable Dataset Size: {total_universe}")

Unique movies in our base financial tables: 7908

Out of 722,000 rows, meta_data has 10027 financially valid movies.
👉 NET NEW movies added by meta_data: 5093

Out of 4,803 rows, tmdb_data has 3228 financially valid movies.
👉 NET NEW movies added by tmdb_data: 46

🚀 Absolute Maximum Viable Dataset Size: 13047


### 2. The Merging Strategy: Outer Join

In [ ]:
import numpy as np

# 1. Pre-Merge Zero Purge
tmdb_data['budget'] = tmdb_data['budget'].replace(0, np.nan)
tmdb_data['revenue'] = tmdb_data['revenue'].replace(0, np.nan)
meta_data['budget'] = meta_data['budget'].replace(0, np.nan)
meta_data['revenue'] = meta_data['revenue'].replace(0, np.nan)
data2['Lifetime Gross'] = data2['Lifetime Gross'].replace(0, np.nan)

/tmp/ipykernel_422/58859257.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_data['budget'] = meta_data['budget'].replace(0, np.nan)
/tmp/ipykernel_422/58859257.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta_data['revenue'] = meta_data['revenue'].replace(0, np.nan)


In [ ]:
# 2. The Outer Join
enriched = pd.merge(movie_franchises, financial_data, on='movie_id', how='outer', suffixes=('_base', '_fin'))
enriched = pd.merge(enriched, tmdb_data, on='movie_id', how='outer', suffixes=('', '_tmdb'))
enriched = pd.merge(enriched, meta_data, on='movie_id', how='outer', suffixes=('', '_meta'))
enriched = pd.merge(enriched, data2, on='movie_id', how='outer')

### 3. Establishing the Single Source of Truth (SSOT)

In [ ]:
# 3. Financial Source of Truth (SSOT) - Using .get() for absolute safety
# Fetch BUDGET columns (if they don't exist, return NaNs)
b_prod = enriched.get('production_budget', pd.Series(np.nan, index=enriched.index))
b_base = enriched.get('budget', pd.Series(np.nan, index=enriched.index)) # It stayed 'budget'!
b_tmdb = enriched.get('budget_tmdb', pd.Series(np.nan, index=enriched.index))
b_meta = enriched.get('budget_meta', pd.Series(np.nan, index=enriched.index))

# Fetch REVENUE columns
r_world = enriched.get('worldwide_gross', pd.Series(np.nan, index=enriched.index))
r_base = enriched.get('gross', pd.Series(np.nan, index=enriched.index)) # It stayed 'gross'!
r_tmdb = enriched.get('revenue', pd.Series(np.nan, index=enriched.index))
r_meta = enriched.get('revenue_meta', pd.Series(np.nan, index=enriched.index))
r_data2 = enriched.get('Lifetime Gross', pd.Series(np.nan, index=enriched.index))

# Coalesce Strategy
enriched['ultimate_budget'] = b_prod.combine_first(b_base).combine_first(b_tmdb).combine_first(b_meta)
enriched['ultimate_revenue'] = r_world.combine_first(r_base).combine_first(r_tmdb).combine_first(r_meta).combine_first(r_data2)

### 4. The Financial North Star (The Great Purge)

In [ ]:
# 4. The Great Purge
enriched_clean = enriched.dropna(subset=['ultimate_budget', 'ultimate_revenue']).copy()
print(f"🔥 FINAL Dataset Size: {len(enriched_clean)} movies ready for Phase II!")

🔥 FINAL Dataset Size: 11361 movies ready for Phase II!


In [ ]:
# Cleanup redundant financial columns
cols_to_drop = [
    'production_budget', 'budget', 'budget_tmdb', 'budget_meta',
    'worldwide_gross', 'gross', 'revenue', 'revenue_meta', 'Lifetime Gross'
]
enriched_clean = enriched_clean.drop(columns=[c for c in cols_to_drop if c in enriched_clean.columns])

---

In [ ]:
# Check for data types and NA's
quick_column_summary(enriched_clean, 'enriched_clean')


📋 Column Summary for `enriched_clean`



,Column,Data Type,NA Count,% Missing
0,name,object,5538,48.75
1,rating,object,5556,48.90
2,genre,object,5538,48.75
3,year_base,float64,5538,48.75
4,released,object,5540,48.76
...,...,...,...,...
98,VoteAvg,float64,11302,99.48
99,VoteCount,float64,11302,99.48
100,FranchiseID,float64,11302,99.48
101,ultimate_budget,float64,0,0.00


In [ ]:
# Check the shape
enriched_clean.shape

(11361, 103)

In [ ]:
# Check how it looks
enriched_clean.head()

,name,rating,genre,year_base,released,imdb_score,votes,director,writer,star,...,Studio,Rating,Runtime,Budget,ReleaseDate,VoteAvg,VoteCount,FranchiseID,ultimate_budget,ultimate_revenue
165,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6300000.0,13416285.0
234,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500000.0,0.0
455,'71,R,Action,2014.0,"October 10, 2014 (United Kingdom)",7.2,53000.0,Yann Demange,Gregory Burke,Jack O'Connell,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11000000.0,3062178.0
620,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0
621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7500000.0,60722734.0


In [ ]:
# --- YEAR CHECK FOR PHASE II ---
year_cols = [col for col in enriched_clean.columns if 'year' in col.lower() or 'date' in col.lower() or 'released' in col.lower()]
print("\nPotential Year columns:", year_cols)

if year_cols:
    quick_column_summary(enriched_clean[year_cols], 'Year Columns Check')


Potential Year columns: ['year_base', 'released', 'year_fin', 'release_date', 'release_date_tmdb', 'release_date_meta', 'Year', 'ReleaseDate']

📋 Column Summary for `Year Columns Check`



,Column,Data Type,NA Count,% Missing
0,year_base,float64,5538,48.75
1,released,object,5540,48.76
2,year_fin,float64,9762,85.93
3,release_date,object,9762,85.93
4,release_date_tmdb,object,7269,63.98
5,release_date_meta,object,540,4.75
6,Year,object,11301,99.47
7,ReleaseDate,object,11302,99.48


# II. Financial Standardization (Macroeconomics)

In [ ]:
# --- PHASE II: FINANCIAL STANDARDIZATION (MACROECONOMICS) ---
import numpy as np
import pandas as pd

print(f"Starting Phase II with {len(enriched_clean)} movies...")
df_finance = enriched_clean.copy()

# 1. Establish an SSOT for the Year
# We pull every possible column that might contain release timing information
y_base = df_finance.get('year', pd.Series(np.nan, index=df_finance.index)).astype(str)
date_tmdb = df_finance.get('release_date', pd.Series(np.nan, index=df_finance.index)).astype(str)
date_base = df_finance.get('released', pd.Series(np.nan, index=df_finance.index)).astype(str)

# Extract just the 4-digit year from all these sources using Regex
y_base_clean = y_base.str.extract(r'(\d{4})')[0]
date_tmdb_clean = date_tmdb.str.extract(r'(\d{4})')[0]
date_base_clean = date_base.str.extract(r'(\d{4})')[0]

# Cascade them together into our ultimate source of truth
df_finance['ultimate_year'] = y_base_clean.combine_first(date_tmdb_clean).combine_first(date_base_clean)

# 2. The Final Purge & Conversion
# Drop rows that still couldn't find a year across any column
df_finance = df_finance.dropna(subset=['ultimate_year'])

# Now safely convert to integer!
df_finance['ultimate_year'] = df_finance['ultimate_year'].astype(int)

print(f"Movies after establishing the ultimate_year: {len(df_finance)}")

# 3. Macroeconomic Data (US Consumer Price Index Base 2024)
cpi_data = {
    1980: 82.4, 1981: 90.9, 1982: 96.5, 1983: 99.6, 1984: 103.9,
    1985: 107.6, 1986: 109.6, 1987: 113.6, 1988: 118.3, 1989: 124.0,
    1990: 130.7, 1991: 136.2, 1992: 140.3, 1993: 144.5, 1994: 148.2,
    1995: 152.4, 1996: 156.9, 1997: 160.5, 1998: 163.0, 1999: 166.6,
    2000: 172.2, 2001: 177.1, 2002: 179.9, 2003: 184.0, 2004: 188.9,
    2005: 195.3, 2006: 201.6, 2007: 207.3, 2008: 215.3, 2009: 214.5,
    2010: 218.1, 2011: 224.9, 2012: 229.6, 2013: 233.0, 2014: 236.7,
    2015: 237.0, 2016: 240.0, 2017: 245.1, 2018: 251.1, 2019: 255.7,
    2020: 258.8, 2021: 271.0, 2022: 292.7, 2023: 304.7, 2024: 313.7
}
cpi_2024 = 313.7

def adjust_for_inflation(row, col_name):
    year = row['ultimate_year']
    amount = row[col_name]
    if year not in cpi_data:
        return amount
    return amount * (cpi_2024 / cpi_data[year])

# 4. Apply the Standardizations
df_finance['real_budget'] = df_finance.apply(lambda x: adjust_for_inflation(x, 'ultimate_budget'), axis=1)
df_finance['real_revenue'] = df_finance.apply(lambda x: adjust_for_inflation(x, 'ultimate_revenue'), axis=1)

# 5. Formulate the Target
df_finance['real_profit'] = df_finance['real_revenue'] - df_finance['real_budget']
df_finance['is_profitable'] = (df_finance['real_profit'] > 0).astype(int)

print("\nPhase II Complete! Financial targets formulated.")

# Quick sanity check
quick_column_summary(df_finance[['ultimate_year', 'real_budget', 'real_revenue', 'real_profit', 'is_profitable']], 'Inflation Adjusted Financials')

Starting Phase II with 11361 movies...
Movies after establishing the ultimate_year: 6219

Phase II Complete! Financial targets formulated.

📋 Column Summary for `Inflation Adjusted Financials`



,Column,Data Type,NA Count,% Missing
0,ultimate_year,int64,0,0.0
1,real_budget,float64,0,0.0
2,real_revenue,float64,0,0.0
3,real_profit,float64,0,0.0
4,is_profitable,int64,0,0.0


## Data Drop Explanation: Data Funnel & Attrition Report

At first glance, moving from a raw dataset of over 722,000 movies down to a final modeling dataset of **6,177 movies** appears to be a massive loss of data. However, this attrition is a deliberate and necessary outcome of our strict financial methodology.

In predictive financial modeling, **data quality strictly supersedes data quantity**. A model trained on millions of rows of incomplete or fabricated financial data will fail spectacularly in production.

Here is the step-by-step breakdown of our Data Funnel, explaining exactly why movies were removed from the pipeline:

### 1. The "Indie & Obscure" Filter (The Great Purge)
* **What Happened:** The vast majority of the 722,000+ movies in the TMDB metadata dataset are student films, obscure international releases, unreleased projects, or straight-to-DVD B-movies.
* **Why they were dropped:** These movies simply do not have public, verified budget or revenue numbers. Our algorithm requires a known `ultimate_budget` and `ultimate_revenue` to calculate ROI. Movies with missing financials (or those padded with `$0` placeholders) were aggressively purged to prevent the model from learning from noise.

### 2. The "Hidden Zero" Purge
* **What Happened:** We actively converted thousands of `$0` budget and revenue values back to `NaN` before dropping them.
* **Why they were dropped:** Many web scrapers default to `$0` when they cannot locate financial data. Allowing a `$0` budget into an ROI calculation creates an infinite return, which would immediately break the mathematical integrity of the target variable.

### 3. The "Lost in Time" Filter (Phase II Attrition)
* **What Happened:** Approximately 5,000 remaining movies were dropped during Phase II.
* **Why they were dropped:** These movies survived the financial purge but lacked a valid, parsable 4-digit `release_date` or `year` across all datasets. Because Phase II requires adjusting nominal dollars to **Real 2024 Dollars** using historical Consumer Price Index (CPI) rates, knowing the exact release year is mathematically non-negotiable. We cannot impute (guess) the year without corrupting the macroeconomic adjustment.

### The Result: A Pristine Financial Universe
The resulting **6,177 movies** represent the absolute highest-quality subset of the global movie market. They are mathematically sound, historically adjusted, and completely free of the structural errors that plague massive, uncurated internet datasets. This pristine dataset is now ready for Feature Engineering.

---

# III. Feature Engineering (Signals & NLP)

## Introduction

With a mathematically sound financial target (`is_profitable`) established, we must translate our raw categorical data into predictive business signals based on the **R.I.C.E. Framework**.

1. **IP & Franchise Power (`is_sequel`):** Franchises represent established market reach and lower customer acquisition costs. We extract this by scanning plot keywords and movie titles.
2. **Historical Authority ("Win Rates"):** An algorithm cannot understand a Director's name. We use **Target Encoding** to convert names (Directors, Stars, Writers) into a mathematical probability score (e.g., this director historically generates a positive ROI 75% of the time).
3. **Scale (`is_major_studio`):** Major studios have distribution pipelines and marketing budgets that independent studios do not. We flag the "Big 6" studios to proxy this market advantage.

## Feature Engineering (Part 1: Confidence & IP)

In [ ]:
# --- PHASE III: FEATURE ENGINEERING (Part 1: Confidence & IP) ---
import pandas as pd

print(f"Engineering features for {len(df_finance)} movies...")
df_features = df_finance.copy()

# 1. IP / Franchise Power (is_sequel)
# We search both the 'keywords' and the movie 'name' for franchise indicators
sequel_keywords = ['sequel', 'trilogy', 'prequel', 'spin off', 'remake', 'franchise', 'shared universe']

def check_ip_status(row):
    # Check keyword tags
    kw = str(row.get('keywords', '')).lower()
    for word in sequel_keywords:
        if word in kw:
            return 1

    # Heuristic: Check if title has numbering (e.g., "Iron Man 2", "Part II", "Vol.")
    title = str(row.get('name', '')).lower()
    if ' 2' in title or ' 3' in title or ' part ' in title or ' vol' in title:
        return 1

    return 0

df_features['is_sequel'] = df_features.apply(check_ip_status, axis=1)

# 2. Historical Authority (Target Encoding for Win Rates)
# We calculate the historical probability of success for the creative team.
def calculate_win_rates(df, col_name):
    # Calculate the mean of 'is_profitable' for each person
    win_rates = df.groupby(col_name)['is_profitable'].mean()

    # Map the win rate back to the dataframe
    # If a person is missing (NaN), we assign them the baseline global win rate
    global_win_rate = df['is_profitable'].mean()
    return df[col_name].map(win_rates).fillna(global_win_rate)

df_features['director_win_rate'] = calculate_win_rates(df_features, 'director')
df_features['star_win_rate'] = calculate_win_rates(df_features, 'star')
df_features['writer_win_rate'] = calculate_win_rates(df_features, 'writer')

# 3. Studio Scale (is_major_studio)
# The "Big" historic studios that guarantee massive distribution and marketing reach
major_studios = ['Warner Bros.', 'Universal Pictures', 'Columbia Pictures',
                 'Paramount Pictures', 'Twentieth Century Fox', 'Walt Disney Pictures',
                 'New Line Cinema']

df_features['is_major_studio'] = df_features['company'].apply(
    lambda x: 1 if pd.notna(x) and any(studio in str(x) for studio in major_studios) else 0
)

print("✅ Phase III (Part 1) Complete: Confidence & Scale Signals Engineered.")

# Let's verify our new features!
quick_column_summary(
    df_features[['is_sequel', 'director_win_rate', 'star_win_rate', 'writer_win_rate', 'is_major_studio']],
    'Engineered R.I.C.E. Signals'
)

Engineering features for 6219 movies...
✅ Phase III (Part 1) Complete: Confidence & Scale Signals Engineered.

📋 Column Summary for `Engineered R.I.C.E. Signals`



,Column,Data Type,NA Count,% Missing
0,is_sequel,int64,0,0.0
1,director_win_rate,float64,0,0.0
2,star_win_rate,float64,0,0.0
3,writer_win_rate,float64,0,0.0
4,is_major_studio,int64,0,0.0


## Feature Engineering (Part 2: NLP & Reach Signals)

In [ ]:
# --- PHASE III: FEATURE ENGINEERING (Part 2: NLP & Reach Signals) ---
from sklearn.feature_extraction.text import CountVectorizer

print(f"Applying NLP to {len(df_features)} movies...")

# 1. Clean the Overview Text
# Fill missing plot summaries with an empty string so the vectorizer doesn't crash
df_features['overview'] = df_features['overview'].fillna('')

# 2. Configure the NLP Vectorizer
# We want the top 30 most common, meaningful words from the plots.
# 'stop_words=english' removes useless words like "the", "and", "is".
vectorizer = CountVectorizer(stop_words='english', max_features=30)

# 3. Fit and Transform the Text
# This creates a mathematical matrix of word counts
word_matrix = vectorizer.fit_transform(df_features['overview'])

# 4. Convert Matrix to DataFrame Columns
# Get the actual words the vectorizer found
nlp_words = vectorizer.get_feature_names_out()

# Add a prefix so we know these are plot themes, not regular columns
nlp_columns = [f"theme_{word}" for word in nlp_words]

# Convert the matrix into a pandas dataframe and align its index with our main dataset
df_nlp = pd.DataFrame(word_matrix.toarray(), columns=nlp_columns, index=df_features.index)

# 5. Join the NLP Features to our Main Dataset
df_model_ready = pd.concat([df_features, df_nlp], axis=1)

print(f"✨ Extracted Top 30 Plot Themes: {', '.join(nlp_words)}")

# --- FINAL CLEANUP BEFORE EXPORT ---

# 6. Drop Text & Leakage Columns
# We must drop raw text, names, and post-release metrics before modeling
cols_to_drop = [
    'name', 'director', 'writer', 'star', 'company', 'country', 'released',
    'overview', 'keywords', 'tagline', 'homepage', 'cast', 'crew', 'recommendations',
    'imdb_score', 'votes', 'vote_average', 'vote_count', 'popularity',
    'ultimate_budget', 'ultimate_revenue', 'real_budget', 'real_revenue', 'real_profit'
    # Notice we drop 'real_profit' because our target is 'is_profitable'
]

# Only drop the columns that actually exist in the dataframe to avoid errors
existing_cols_to_drop = [c for c in cols_to_drop if c in df_model_ready.columns]
df_final = df_model_ready.drop(columns=existing_cols_to_drop)

print(f"\n🚀 Final Dataset Shape for Modeling: {df_final.shape}")

Applying NLP to 6219 movies...
✨ Extracted Top 30 Plot Themes: city, family, father, finds, friend, friends, help, high, home, life, lives, love, man, new, old, school, son, soon, story, team, time, town, war, way, wife, woman, world, year, years, young

🚀 Final Dataset Shape for Modeling: (6219, 119)


# Export Final Dataset

In [ ]:
# 7. EXPORT TO CSV OUR FINAL PREPROCESSED DATASET
file_name = 'greenlight_model_data.csv'
df_final.to_csv(file_name, index=False)

from google.colab import files
try:
    files.download(file_name)
    print(f"✅ Successfully downloaded '{file_name}'!")
except Exception as e:
    print(f"✅ Saved '{file_name}' to local Colab storage.")